# Warfarin preprocessing V2 — cleaned candidate feature sets

This notebook is the active V2 preprocessing checkpoint. It keeps the improved missing-value handling from V2, but trims the experiment space to the feature sets that survived the first benchmark round.

Active feature sets generated here:

1. `v2_pharmacogenetic_augmented_regression_hw` — strongest current candidate.
2. `v2_strict_regression_hw_full_dummies` — best non-baseline-augmented V2 feature set.
3. `v2_strict_knn_hw_full_dummies` — compact reference against the original height/weight imputation style.

Retired from the active flow: group-median height/weight, drop-first dummy encoding, expanded target-INR/indication features, and IWPC-minimal feature sets. Those experiments remain recoverable from earlier commits, but are no longer generated by the current notebook.


## Height / weight imputation overview

The original preprocessing used KNN imputation on only two columns:

\[
X_{hw} = [\text{Height}, \text{Weight}]
\]

The values were standardized, imputed with `KNNImputer(n_neighbors=5)`, then inverse transformed. That is useful when one of height/weight is present, but when **both** are missing the imputer has almost no patient-specific signal and falls back toward global averages.

V2 keeps that old-style variant for comparison, but also creates two stronger alternatives:

1. **Group median imputation** using demographic groups such as gender, race, and age bucket.
2. **Regression-style multivariate imputation** using height, weight, age, gender, race, ethnicity, enzyme-inducer status, amiodarone status, and selected clinical/binary variables.

The regression imputer is the preferred V2 default because it uses patient context rather than only the two anthropometric columns.

In [ ]:
from pathlib import Path
import os
import sys
import json

import numpy as np
import pandas as pd

from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import KNNImputer, IterativeImputer
from sklearn.linear_model import BayesianRidge
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)


def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for candidate in [start] + list(start.parents):
        if (candidate / "data" / "warfarin.csv").exists() and (candidate / "src").exists():
            return candidate
    raise FileNotFoundError("Could not find repository root containing data/warfarin.csv and src/.")


REPO_ROOT = find_repo_root()
os.chdir(REPO_ROOT)

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

RAW_PATH = REPO_ROOT / "data" / "warfarin.csv"
OUTPUT_DIR = REPO_ROOT / "output" / "preprocess_v2"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_COL = "Therapeutic Dose of Warfarin"
TRUE_DOSE_COL = "Therapeutic Dose of Warfarin__mg_week"
DOSE_BINS = [0, 20.9999, 49, np.inf]
DOSE_LABELS = [0, 1, 2]

print("Repo root:", REPO_ROOT)
print("Raw data path:", RAW_PATH)
print("V2 output directory:", OUTPUT_DIR)

## Configuration

The output file names are unchanged so downstream notebooks can keep using the same V2 paths. The feature manifest now contains only the serious candidates.


In [ ]:
ID_COLUMNS = ["PharmGKB Subject ID"]
EMPTY_COLUMNS = ["Unnamed: 63", "Unnamed: 64", "Unnamed: 65"]

POST_TREATMENT_OR_OUTCOME_COLUMNS = [
    "Subject Reached Stable Dose of Warfarin",
    "INR on Reported Therapeutic Dose of Warfarin",
]

FREE_TEXT_COLUMNS = [
    "Comorbidities",
    "Medications",
]

# Columns where the appendix / IWPC algorithm explicitly assumes unknown use as not used
# for formula purposes. We also use these cleaned 0/1 statuses to define enzyme inducer status.
FORMULA_ZERO_IF_UNKNOWN_COLUMNS = [
    "Amiodarone (Cordarone)",
    "Carbamazepine (Tegretol)",
    "Phenytoin (Dilantin)",
    "Rifampin or Rifampicin",
]

# Other 0/1 clinical/medication columns where missingness is better treated as an explicit category.
BINARY_UNKNOWN_COLUMNS = [
    "Diabetes",
    "Congestive Heart Failure and/or Cardiomyopathy",
    "Valve Replacement",
    "Aspirin",
    "Acetaminophen or Paracetamol (Tylenol)",
    "Was Dose of Acetaminophen or Paracetamol (Tylenol) >1300mg/day",
    "Simvastatin (Zocor)",
    "Atorvastatin (Lipitor)",
    "Fluvastatin (Lescol)",
    "Lovastatin (Mevacor)",
    "Pravastatin (Pravachol)",
    "Rosuvastatin (Crestor)",
    "Cerivastatin (Baycol)",
    "Sulfonamide Antibiotics",
    "Macrolide Antibiotics",
    "Anti-fungal Azoles",
    "Herbal Medications, Vitamins, Supplements",
    "Current Smoker",
]

GENOTYPE_COLUMNS = [
    "Cyp2C9 genotypes",
    "Genotyped QC Cyp2C9*2",
    "Genotyped QC Cyp2C9*3",
    "Combined QC CYP2C9",
    "VKORC1 genotype: -1639 G>A (3673); chr16:31015190; rs9923231; C/T",
    "VKORC1 QC genotype: -1639 G>A (3673); chr16:31015190; rs9923231; C/T",
    "VKORC1 genotype: 497T>G (5808); chr16:31013055; rs2884737; A/C",
    "VKORC1 QC genotype: 497T>G (5808); chr16:31013055; rs2884737; A/C",
    "VKORC1 genotype: 1173 C>T(6484); chr16:31012379; rs9934438; A/G",
    "VKORC1 QC genotype: 1173 C>T(6484); chr16:31012379; rs9934438; A/G",
    "VKORC1 genotype: 1542G>C (6853); chr16:31012010; rs8050894; C/G",
    "VKORC1 QC genotype: 1542G>C (6853); chr16:31012010; rs8050894; C/G",
    "VKORC1 genotype: 3730 G>A (9041); chr16:31009822; rs7294;  A/G",
    "VKORC1 QC genotype: 3730 G>A (9041); chr16:31009822; rs7294;  A/G",
    "VKORC1 genotype: 2255C>T (7566); chr16:31011297; rs2359612; A/G",
    "VKORC1 QC genotype: 2255C>T (7566); chr16:31011297; rs2359612; A/G",
    "VKORC1 genotype: -4451 C>A (861); Chr16:31018002; rs17880887; A/C",
    "VKORC1 QC genotype: -4451 C>A (861); Chr16:31018002; rs17880887; A/C",
    "CYP2C9 consensus",
    "VKORC1 -1639 consensus",
    "VKORC1 497 consensus",
    "VKORC1 1173 consensus",
    "VKORC1 1542 consensus",
    "VKORC1 3730 consensus",
    "VKORC1 2255 consensus",
    "VKORC1 -4451 consensus",
]

AGE_TO_DECADE = {
    "0 - 9": 0,
    "10 - 19": 1,
    "20 - 29": 2,
    "30 - 39": 3,
    "40 - 49": 4,
    "50 - 59": 5,
    "60 - 69": 6,
    "70 - 79": 7,
    "80 - 89": 8,
    "90+": 9,
}

INDICATION_MAP = {
    "1": "DVT",
    "2": "PE",
    "3": "Afib_or_flutter",
    "4": "Heart_Valve",
    "5": "Cardiomyopathy_or_LV_Dilation",
    "6": "Stroke",
    "7": "Post_Orthopedic",
    "8": "Other",
}

## Load raw data and inspect missingness

In [ ]:
df_raw = pd.read_csv(RAW_PATH)
print("Raw shape:", df_raw.shape)

missing_report_raw = (
    df_raw.isna().sum()
    .rename("missing_count")
    .reset_index()
    .rename(columns={"index": "column"})
)
missing_report_raw["missing_pct"] = missing_report_raw["missing_count"] / len(df_raw)
missing_report_raw = missing_report_raw.sort_values("missing_count", ascending=False)

missing_report_path = OUTPUT_DIR / "warfarin_preprocess_v2_missingness_report_raw.csv"
missing_report_raw.to_csv(missing_report_path, index=False)

print("Saved raw missingness report:", missing_report_path)
missing_report_raw.head(30)

In [ ]:
# Keep only patients with known therapeutic dose, as specified by the project.
df = df_raw[df_raw[TARGET_COL].notna()].copy()
df[TRUE_DOSE_COL] = df[TARGET_COL].astype(float)
df[TARGET_COL] = pd.cut(
    df[TRUE_DOSE_COL],
    bins=DOSE_BINS,
    labels=DOSE_LABELS,
    include_lowest=True,
).astype(int)

print("Shape after dropping unknown therapeutic dose:", df.shape)
print("Dose bucket distribution:")
print(df[TARGET_COL].value_counts().sort_index())

## Height / weight missingness statistics

These counts help us understand why the original two-column KNN imputation is limited. When both height and weight are missing, a two-column KNN imputer cannot use patient-level context.

In [ ]:
height_missing = df["Height (cm)"].isna()
weight_missing = df["Weight (kg)"].isna()

height_weight_missing_stats = pd.DataFrame(
    [
        {"case": "height_present_weight_present", "count": int((~height_missing & ~weight_missing).sum())},
        {"case": "height_missing_weight_present", "count": int((height_missing & ~weight_missing).sum())},
        {"case": "height_present_weight_missing", "count": int((~height_missing & weight_missing).sum())},
        {"case": "height_missing_weight_missing", "count": int((height_missing & weight_missing).sum())},
    ]
)
height_weight_missing_stats["pct"] = height_weight_missing_stats["count"] / len(df)
height_weight_missing_stats_path = OUTPUT_DIR / "warfarin_preprocess_v2_height_weight_missingness.csv"
height_weight_missing_stats.to_csv(height_weight_missing_stats_path, index=False)

print("Saved height/weight missingness stats:", height_weight_missing_stats_path)
height_weight_missing_stats

## Core cleaning helpers

In [ ]:
def yes_no_unknown(series):
    return series.map({1.0: "Yes", 0.0: "No", 1: "Yes", 0: "No"}).fillna("Unknown")


def parse_target_range_midpoint(value):
    if pd.isna(value):
        return np.nan
    text = str(value).strip().replace(" ", "")
    if "-" not in text:
        try:
            return float(text)
        except ValueError:
            return np.nan
    left, right = text.split("-", 1)
    try:
        return (float(left) + float(right)) / 2
    except ValueError:
        return np.nan


def parse_indication_codes(value):
    if pd.isna(value):
        return []
    text = str(value).replace("or", ";").replace(",", ";")
    parts = [part.strip() for part in text.split(";")]
    return [part for part in parts if part in INDICATION_MAP]


def add_indication_features(frame):
    parsed = frame["Indication for Warfarin Treatment"].apply(parse_indication_codes)
    for code, name in INDICATION_MAP.items():
        frame[f"Indication__{name}"] = parsed.apply(lambda codes, c=code: int(c in codes))
    frame["Indication__Unknown"] = parsed.apply(lambda codes: int(len(codes) == 0))
    frame["Indication__Multiple"] = parsed.apply(lambda codes: int(len(codes) > 1))
    return frame


def bucket_weekly_dose(values):
    return pd.cut(values, bins=DOSE_BINS, labels=DOSE_LABELS, include_lowest=True).astype(int)


def save_json(data, path):
    with open(path, "w", encoding="utf-8") as handle:
        json.dump(data, handle, indent=2)

## Apply V2 cleaning rules

In [ ]:
# Drop empty columns but keep the subject ID in the raw-clean table for traceability.
df = df.drop(columns=[c for c in EMPTY_COLUMNS if c in df.columns], errors="ignore")

# Demographics
for col in ["Gender", "Race", "Ethnicity"]:
    df[col] = df[col].fillna("Unknown")

mode_age_bucket = df["Age"].mode(dropna=True).iloc[0]
df["Age__mode_bucket"] = df["Age"].fillna(mode_age_bucket)
df["Age__bucket_unknown"] = df["Age"].fillna("Unknown")
df["Age__missing"] = df["Age"].isna().astype(int)
df["Age__mode_decade"] = df["Age__mode_bucket"].map(AGE_TO_DECADE).astype(float)

print("Age mode bucket used for numeric age:", mode_age_bucket)
print(df[["Age", "Age__mode_bucket", "Age__bucket_unknown", "Age__mode_decade"]].head())

In [ ]:
# Appendix/IWPC formula statuses: unknown is treated as not used for these formula-specific statuses.
for col in FORMULA_ZERO_IF_UNKNOWN_COLUMNS:
    if col in df.columns:
        df[f"{col}__missing"] = df[col].isna().astype(int)
        df[col] = df[col].fillna(0).astype(float)

# Enzyme inducer status is defined by carbamazepine, phenytoin, or rifampin/rifampicin use.
df["Enzyme inducer status"] = (
    (df["Carbamazepine (Tegretol)"] == 1)
    | (df["Phenytoin (Dilantin)"] == 1)
    | (df["Rifampin or Rifampicin"] == 1)
).astype(float)

# Other binary variables keep Unknown as its own category.
for col in BINARY_UNKNOWN_COLUMNS:
    if col in df.columns:
        df[col] = yes_no_unknown(df[col])

print("Enzyme inducer status value counts:")
print(df["Enzyme inducer status"].value_counts(dropna=False))

## VKORC1 rs9923231 imputation from appendix S4

The appendix gives a decision list for imputing VKORC1 rs9923231 from nearby VKORC1 SNPs and race. V2 keeps that logic, but treats the dataset value `Unknown` as the missing/mixed-race category for the race-gated conditions.

In [ ]:
def impute_vkorc1_rs9923231(frame):
    frame = frame.copy()
    target = "VKORC1 genotype: -1639 G>A (3673); chr16:31015190; rs9923231; C/T"
    rs2359612 = "VKORC1 genotype: 2255C>T (7566); chr16:31011297; rs2359612; A/G"
    rs9934438 = "VKORC1 genotype: 1173 C>T(6484); chr16:31012379; rs9934438; A/G"
    rs8050894 = "VKORC1 genotype: 1542G>C (6853); chr16:31012010; rs8050894; C/G"

    before_missing = int(frame[target].isna().sum())
    race_blocks_race_gated_imputation = frame["Race"].isin([
        "Black or African American",
        "Missing or Mixed Race",
        "Unknown",
    ])
    race_gated = ~race_blocks_race_gated_imputation

    rules = [
        (race_gated & (frame[rs2359612] == "C/C"), "G/G"),
        (race_gated & (frame[rs2359612] == "T/T"), "A/A"),
        (race_gated & (frame[rs2359612] == "C/T"), "A/G"),
        ((frame[rs9934438] == "C/C"), "G/G"),
        ((frame[rs9934438] == "T/T"), "A/A"),
        ((frame[rs9934438] == "C/T"), "A/G"),
        (race_gated & (frame[rs8050894] == "G/G"), "G/G"),
        (race_gated & (frame[rs8050894] == "C/C"), "A/A"),
        (race_gated & (frame[rs8050894] == "C/G"), "A/G"),
    ]

    for condition, value in rules:
        frame.loc[condition & frame[target].isna(), target] = value

    after_rules_missing = int(frame[target].isna().sum())
    frame[target] = frame[target].fillna("Unknown")
    after_final_missing = int(frame[target].isna().sum())

    report = {
        "target_column": target,
        "missing_before": before_missing,
        "missing_after_decision_rules": after_rules_missing,
        "missing_after_unknown_fill": after_final_missing,
        "imputed_by_decision_rules": before_missing - after_rules_missing,
    }
    return frame, report


df, vkorc1_report = impute_vkorc1_rs9923231(df)

# Fill remaining genotype and consensus missingness as Unknown.
for col in GENOTYPE_COLUMNS:
    if col in df.columns:
        df[col] = df[col].fillna("Unknown")

vkorc1_report_path = OUTPUT_DIR / "warfarin_preprocess_v2_vkorc1_imputation_report.json"
save_json(vkorc1_report, vkorc1_report_path)

print("Saved VKORC1 imputation report:", vkorc1_report_path)
print(vkorc1_report)

## Retired expanded clinical-context features

Earlier V2 experiments engineered target-INR and indication features. They did not improve the RidgeLinUCB leaderboard enough to justify keeping them in the active experiment path, so the cleaned V2 flow drops those raw columns before one-hot encoding and does not generate expanded feature sets.


In [ ]:
# Intentionally no active target-INR / indication feature engineering here.
# These columns are dropped before one-hot encoding in the modeling-table cell.
print("Expanded target-INR / indication features are retired in the cleaned V2 flow.")


## Height / weight imputation variants

In [ ]:
# Missingness flags are useful even after imputation.
df["Height (cm)__missing"] = df["Height (cm)"].isna().astype(int)
df["Weight (kg)__missing"] = df["Weight (kg)"].isna().astype(int)
df["HeightWeight__both_missing"] = (
    df["Height (cm)"].isna() & df["Weight (kg)"].isna()
).astype(int)

# Variant 1: old-style KNN on only height and weight.
# Kept only as a reference feature set, because regression imputation performed better overall.
hw_scaler = StandardScaler()
hw_scaled = hw_scaler.fit_transform(df[["Height (cm)", "Weight (kg)"]])
hw_knn = hw_scaler.inverse_transform(KNNImputer(n_neighbors=5).fit_transform(hw_scaled))
df["Height (cm)__knn_hw"] = hw_knn[:, 0]
df["Weight (kg)__knn_hw"] = hw_knn[:, 1]

# Variant 2: regression-style multivariate imputation using patient context.
# This is the active height/weight strategy for the strongest feature sets.
regression_auxiliary_columns = [
    "Age__mode_decade",
    "Gender",
    "Race",
    "Ethnicity",
    "Enzyme inducer status",
    "Amiodarone (Cordarone)",
    "Carbamazepine (Tegretol)",
    "Phenytoin (Dilantin)",
    "Rifampin or Rifampicin",
    "Diabetes",
    "Congestive Heart Failure and/or Cardiomyopathy",
    "Valve Replacement",
    "Current Smoker",
]
regression_auxiliary_columns = [col for col in regression_auxiliary_columns if col in df.columns]

imputation_frame = pd.concat(
    [
        df[["Height (cm)", "Weight (kg)"]],
        pd.get_dummies(df[regression_auxiliary_columns], dummy_na=False, dtype=float),
    ],
    axis=1,
)

regression_imputer = IterativeImputer(
    estimator=BayesianRidge(),
    max_iter=25,
    random_state=42,
    initial_strategy="median",
    sample_posterior=False,
)
regression_imputed = pd.DataFrame(
    regression_imputer.fit_transform(imputation_frame),
    columns=imputation_frame.columns,
    index=df.index,
)

df["Height (cm)__regression"] = regression_imputed["Height (cm)"]
df["Weight (kg)__regression"] = regression_imputed["Weight (kg)"]

hw_variant_summary = df[[
    "Height (cm)", "Weight (kg)",
    "Height (cm)__knn_hw", "Weight (kg)__knn_hw",
    "Height (cm)__regression", "Weight (kg)__regression",
]].describe().T

hw_variant_summary_path = OUTPUT_DIR / "warfarin_preprocess_v2_height_weight_variant_summary.csv"
hw_variant_summary.to_csv(hw_variant_summary_path)

print("Saved height/weight variant summary:", hw_variant_summary_path)
hw_variant_summary


## Clinical and pharmacogenetic dose estimates as optional warm-start features

These are deterministic functions of patient covariates, not the true therapeutic dose. They are useful later as optional warm-start / prior features for contextual bandits.

In [ ]:
def race_coefficient_clinical(race):
    if race == "Asian":
        return -0.6752
    if race == "Black or African American":
        return 0.4060
    if race in ["Unknown", "Missing or Mixed Race"]:
        return 0.0443
    return 0.0


def race_coefficient_pharmacogenetic(race):
    if race == "Asian":
        return -0.1092
    if race == "Black or African American":
        return -0.2760
    if race in ["Unknown", "Missing or Mixed Race"]:
        return -0.1032
    return 0.0


def cyp2c9_coefficient(genotype):
    return {
        "*1/*2": -0.5211,
        "*1/*3": -0.9357,
        "*2/*2": -1.0616,
        "*2/*3": -1.9206,
        "*3/*3": -2.3312,
        "Unknown": -0.2188,
    }.get(genotype, 0.0)


def vkorc1_coefficient(genotype):
    return {
        "A/A": -1.6974,
        "A/G": -0.8677,
        "Unknown": -0.4854,
    }.get(genotype, 0.0)


def add_baseline_estimates(frame, suffix, height_col, weight_col):
    vkorc1_col = "VKORC1 genotype: -1639 G>A (3673); chr16:31015190; rs9923231; C/T"
    race_clinical = frame["Race"].apply(race_coefficient_clinical)
    race_pharma = frame["Race"].apply(race_coefficient_pharmacogenetic)
    cyp_coeff = frame["Cyp2C9 genotypes"].apply(cyp2c9_coefficient)
    vkorc1_coeff = frame[vkorc1_col].apply(vkorc1_coefficient)

    clinical_sqrt = (
        4.0376
        - 0.2546 * frame["Age__mode_decade"]
        + 0.0118 * frame[height_col]
        + 0.0134 * frame[weight_col]
        + race_clinical
        + 1.2799 * frame["Enzyme inducer status"]
        - 0.5695 * frame["Amiodarone (Cordarone)"]
    )

    pharma_sqrt = (
        5.6044
        - 0.2614 * frame["Age__mode_decade"]
        + 0.0087 * frame[height_col]
        + 0.0128 * frame[weight_col]
        + race_pharma
        + 1.1816 * frame["Enzyme inducer status"]
        - 0.5503 * frame["Amiodarone (Cordarone)"]
        + cyp_coeff
        + vkorc1_coeff
    )

    frame[f"Clinical Dose Estimate__{suffix}"] = clinical_sqrt ** 2
    frame[f"Pharmacogenetic Dose Estimate__{suffix}"] = pharma_sqrt ** 2
    frame[f"Clinical Dose Bucket__{suffix}"] = bucket_weekly_dose(frame[f"Clinical Dose Estimate__{suffix}"])
    frame[f"Pharmacogenetic Dose Bucket__{suffix}"] = bucket_weekly_dose(frame[f"Pharmacogenetic Dose Estimate__{suffix}"])
    return frame


for suffix, h_col, w_col in [
    ("knn_hw", "Height (cm)__knn_hw", "Weight (kg)__knn_hw"),
    ("regression_hw", "Height (cm)__regression", "Weight (kg)__regression"),
]:
    df = add_baseline_estimates(df, suffix, h_col, w_col)

baseline_accuracy_rows = []
for suffix in ["knn_hw", "regression_hw"]:
    for baseline in ["Clinical", "Pharmacogenetic"]:
        pred_col = f"{baseline} Dose Bucket__{suffix}"
        baseline_accuracy_rows.append({
            "baseline": baseline,
            "height_weight_variant": suffix,
            "accuracy": float((df[pred_col] == df[TARGET_COL]).mean()),
        })

baseline_accuracy = pd.DataFrame(baseline_accuracy_rows)
baseline_accuracy_path = OUTPUT_DIR / "warfarin_preprocess_v2_baseline_accuracy_by_hw_variant.csv"
baseline_accuracy.to_csv(baseline_accuracy_path, index=False)

print("Saved baseline accuracy by height/weight variant:", baseline_accuracy_path)
baseline_accuracy


## Build V2 raw-clean and modeling tables

The raw-clean table preserves readable columns and engineered variants. The modeling table is numeric after one-hot encoding, with outcome/post-treatment leakage columns removed.

In [ ]:
raw_clean_path = OUTPUT_DIR / "warfarin_preprocess_v2_raw_clean_table.csv"
df.to_csv(raw_clean_path, index=False)
print("Saved raw-clean V2 table:", raw_clean_path)
print("Raw-clean shape:", df.shape)

In [ ]:
# Columns that should not be one-hot encoded into the modeling table.
# We keep engineered replacements instead.
columns_to_drop_before_encoding = [
    *ID_COLUMNS,
    *POST_TREATMENT_OR_OUTCOME_COLUMNS,
    *FREE_TEXT_COLUMNS,
    "Age",
    "Age__mode_bucket",
    "Height (cm)",
    "Weight (kg)",
    "Indication for Warfarin Treatment",
    "Target INR",
    "Estimated Target INR Range Based on Indication",
    "Estimated Target INR Range Based on Indication__midpoint",
]
columns_to_drop_before_encoding = [c for c in columns_to_drop_before_encoding if c in df.columns]

model_source = df.drop(columns=columns_to_drop_before_encoding, errors="ignore").copy()

# Track categorical dummy groups so feature sets can optionally drop one level per group.
categorical_cols = model_source.select_dtypes(include=["object", "category"]).columns.tolist()
category_levels = {
    col: sorted(model_source[col].dropna().astype(str).unique().tolist())
    for col in categorical_cols
}

model_encoded = pd.get_dummies(model_source, columns=categorical_cols, dummy_na=False, dtype=float)

# Ensure bool columns are numeric.
for col in model_encoded.columns:
    if model_encoded[col].dtype == bool:
        model_encoded[col] = model_encoded[col].astype(int)

# A useful intercept column for linear bandits.
model_encoded["Intercept"] = 1.0

modeling_table_path = OUTPUT_DIR / "warfarin_preprocess_v2_modeling_table.csv"
model_encoded.to_csv(modeling_table_path, index=False)

category_levels_path = OUTPUT_DIR / "warfarin_preprocess_v2_category_levels.json"
save_json(category_levels, category_levels_path)

print("Saved V2 modeling table:", modeling_table_path)
print("Saved category levels:", category_levels_path)
print("Modeling shape:", model_encoded.shape)
model_encoded.head()

## Feature-set manifests

The modeling table intentionally contains more columns than any single experiment needs. The feature-set manifest lets later bandit notebooks select a named set of columns.

Every feature set excludes:

- true weekly dose;
- target label as a feature;
- stable-dose outcome column;
- INR on reported therapeutic dose;
- subject identifier and raw free-text fields.

In [ ]:
TARGET_AND_NON_FEATURE_COLUMNS = {
    TARGET_COL,
    TRUE_DOSE_COL,
}

HEIGHT_WEIGHT_VARIANT_COLS = {
    "knn_hw": ["Height (cm)__knn_hw", "Weight (kg)__knn_hw"],
    "regression_hw": ["Height (cm)__regression", "Weight (kg)__regression"],
}
ALL_HEIGHT_WEIGHT_VARIANT_COLS = [col for cols in HEIGHT_WEIGHT_VARIANT_COLS.values() for col in cols]

BASELINE_FEATURE_PATTERNS = [
    "Clinical Dose Estimate__",
    "Pharmacogenetic Dose Estimate__",
    "Clinical Dose Bucket__",
    "Pharmacogenetic Dose Bucket__",
]


def columns_matching_patterns(columns, patterns):
    return [col for col in columns if any(pattern in col for pattern in patterns)]


def base_feature_columns(hw_variant="regression_hw", include_baselines=False):
    features = [col for col in model_encoded.columns if col not in TARGET_AND_NON_FEATURE_COLUMNS]

    selected_hw_cols = set(HEIGHT_WEIGHT_VARIANT_COLS[hw_variant])
    features = [
        col for col in features
        if (col not in ALL_HEIGHT_WEIGHT_VARIANT_COLS) or (col in selected_hw_cols)
    ]

    baseline_cols = set(columns_matching_patterns(features, BASELINE_FEATURE_PATTERNS))
    if include_baselines:
        features = [
            col for col in features
            if (col not in baseline_cols) or col.endswith(f"__{hw_variant}")
        ]
    else:
        features = [col for col in features if col not in baseline_cols]

    return features


feature_sets = {
    "v2_pharmacogenetic_augmented_regression_hw": base_feature_columns("regression_hw", include_baselines=True),
    "v2_strict_regression_hw_full_dummies": base_feature_columns("regression_hw", include_baselines=False),
    "v2_strict_knn_hw_full_dummies": base_feature_columns("knn_hw", include_baselines=False),
}

feature_manifest = pd.DataFrame(
    {"feature_set": name, "feature_name": feature}
    for name, features in feature_sets.items()
    for feature in features
)
feature_summary = (
    feature_manifest.groupby("feature_set")
    .size()
    .reset_index(name="n_features")
    .sort_values("feature_set")
)

feature_manifest_path = OUTPUT_DIR / "warfarin_preprocess_v2_feature_sets.csv"
feature_summary_path = OUTPUT_DIR / "warfarin_preprocess_v2_feature_set_summary.csv"
feature_manifest.to_csv(feature_manifest_path, index=False)
feature_summary.to_csv(feature_summary_path, index=False)

print("Saved feature manifest:", feature_manifest_path)
print("Saved feature-set summary:", feature_summary_path)
feature_summary


## Sanity checks

In [ ]:
# 1. No feature set should contain target/non-feature columns.
for feature_set_name, features in feature_sets.items():
    overlap = sorted(set(features) & TARGET_AND_NON_FEATURE_COLUMNS)
    assert not overlap, f"{feature_set_name} contains non-feature columns: {overlap}"

# 2. No feature set should contain post-treatment/outcome leakage columns.
leakage_terms = [
    "Subject Reached Stable Dose of Warfarin",
    "INR on Reported Therapeutic Dose of Warfarin",
    TRUE_DOSE_COL,
]
for feature_set_name, features in feature_sets.items():
    leakage_hits = [feature for feature in features if any(term in feature for term in leakage_terms)]
    assert not leakage_hits, f"{feature_set_name} contains leakage columns: {leakage_hits[:5]}"

# 3. All feature-set columns should exist in the modeling table and be numeric.
for feature_set_name, features in feature_sets.items():
    missing = [feature for feature in features if feature not in model_encoded.columns]
    assert not missing, f"{feature_set_name} has missing columns: {missing[:5]}"
    non_numeric = model_encoded[features].select_dtypes(exclude=[np.number]).columns.tolist()
    assert not non_numeric, f"{feature_set_name} has non-numeric columns: {non_numeric[:5]}"

# 4. Modeling table should not contain NaNs.
remaining_nulls = int(model_encoded.isna().sum().sum())
assert remaining_nulls == 0, f"Modeling table still contains {remaining_nulls} missing values"

print("All sanity checks passed.")
print("Final modeling table shape:", model_encoded.shape)
print("Feature-set counts:")
print(feature_summary.to_string(index=False))

## How to load the cleaned V2 feature sets later

```python
modeling_df = pd.read_csv("output/preprocess_v2/warfarin_preprocess_v2_modeling_table.csv")
feature_sets = pd.read_csv("output/preprocess_v2/warfarin_preprocess_v2_feature_sets.csv")
feature_cols = feature_sets.loc[
    feature_sets["feature_set"] == "v2_pharmacogenetic_augmented_regression_hw",
    "feature_name",
].tolist()
X = modeling_df[feature_cols]
y = modeling_df["Therapeutic Dose of Warfarin"]
```

The active V2 feature manifest now intentionally stays small. Earlier exploratory feature sets are retired from the active workflow and can be recovered from Git history if needed.
